# Istio + Teleport Workload Identity — Part 2: Off-Cluster VM via mTLS/SPIFFE

This notebook extends [Part 1](demo-walkthrough.ipynb) to demonstrate a **simulated off-cluster VM** authenticating
to the Sock Shop `catalogue` service using a SPIFFE SVID issued by Teleport Workload Identity.

## What This Proves

- A workload *outside* the Kubernetes mesh joins the same zero-trust identity fabric as the in-mesh pods.
- `tbot` on the VM obtains a SPIFFE SVID from Teleport and serves it locally via the SPIFFE Workload API socket.
- Envoy fetches that SVID entirely **in-memory via SDS** — no certificates ever touch disk on the VM.
- The Istio sidecar on `catalogue` trusts the SVID because Teleport is the Istio mesh CA (Scenario A from Part 1).
- An Istio `AuthorizationPolicy` keyed on the VM's SPIFFE ID allows or denies access — editable live, no restarts.

## Architecture

```
┌──────────────────────────────────────────┐
│  Docker container (simulated VM)          │
│                                          │
│  tbot ──► /run/teleport/workload.sock    │
│                    ▲                     │
│           Envoy (SDS) ──────────────┐   │
│           localhost:8080 (proxy)    │   │
└─────────────────────────────────────┼───┘
                                      │ mTLS (Teleport SPIFFE SVID)
                                      ▼
                         Kubernetes cluster
                         ┌──────────────────────────────┐
                         │  catalogue-external LB svc    │
                         │  → pod iptables intercept     │
                         │  → Istio sidecar port 15006   │
                         │    validates SVID             │
                         │    checks AuthorizationPolicy │
                         │  → catalogue app container    │
                         └──────────────────────────────┘
```

## Prerequisites

- Part 1 notebook completed — Istio, tbot DaemonSet, and Sock Shop are deployed and healthy.
- `docker` and `docker compose` are available on this machine.
- `.env` file contains `TELEPORT_TRUST_DOMAIN`.

**Run each cell in order.**

## 0. Notebook Setup

Same helper functions as Part 1, plus environment variable loading.

In [45]:
import subprocess
import os
import re
import shutil
import time
from IPython.display import display, Markdown

REPO_DIR = os.path.dirname(os.path.abspath("demo-walkthrough-part2.ipynb"))


class _RunResult:
    def __init__(self, proc):
        object.__setattr__(self, '_proc', proc)

    def __getattr__(self, name):
        return getattr(object.__getattribute__(self, '_proc'), name)

    def _ipython_display_(self, **kwargs):
        pass


def run(cmd, *, cwd=REPO_DIR, check=True, capture=False, delay=0):
    """Run a shell command, print output, return CompletedProcess."""
    print(f"$ {cmd}")
    result = subprocess.run(
        cmd, shell=True, cwd=cwd,
        capture_output=capture,
        text=True
    )
    if capture:
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd}")
    if delay:
        time.sleep(delay)
    return _RunResult(result)


# Load .env
env_path = os.path.join(REPO_DIR, ".env")
if not os.path.exists(env_path):
    raise FileNotFoundError(".env not found — run Part 1 setup first")

env_vars = {}
with open(env_path) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, _, v = line.partition("=")
            env_vars[k.strip()] = v.strip()

TELEPORT_TRUST_DOMAIN = env_vars.get("TELEPORT_TRUST_DOMAIN", "")
if not TELEPORT_TRUST_DOMAIN or TELEPORT_TRUST_DOMAIN == "your-cluster.teleport.sh":
    raise ValueError("TELEPORT_TRUST_DOMAIN is not set in .env — run Part 1 first")

TELEPORT_PROXY = f"{TELEPORT_TRUST_DOMAIN}:443"

print(f"Trust domain : {TELEPORT_TRUST_DOMAIN}")
print(f"Proxy        : {TELEPORT_PROXY}")
print(f"Repo dir     : {REPO_DIR}")

Trust domain : ellinj.teleport.sh
Proxy        : ellinj.teleport.sh:443
Repo dir     : /Users/jeff/dev/rev-tech/use-cases/mwi/istio-spiffe


## Step 0 — Verify Baseline from Part 1

Confirm that Sock Shop and the tbot DaemonSet are still healthy before we build on top.

In [46]:
print("=== Sock Shop pods (expect 2/2 Running) ===")
run("kubectl get pods -n sock-shop", capture=True)

print("\n=== tbot DaemonSet (expect 1/1 per node) ===")
run("kubectl get pods -n teleport-system", capture=True)

print("\n=== Existing AuthorizationPolicies in sock-shop ===")
run("kubectl get authorizationpolicies -n sock-shop", capture=True)

=== Sock Shop pods (expect 2/2 Running) ===
$ kubectl get pods -n sock-shop
NAME                            READY   STATUS    RESTARTS   AGE
carts-85f899d8cf-g4fr7          2/2     Running   0          5m50s
carts-db-787cb57b5-74r8d        2/2     Running   0          5m49s
catalogue-68c47ffd46-hzqr2      2/2     Running   0          5m50s
catalogue-db-67b98f5dcd-2hwdp   2/2     Running   0          5m50s
front-end-58bd5f4987-h54mj      2/2     Running   0          5m50s
orders-75f5dffd8b-sf5zs         2/2     Running   0          5m49s


=== tbot DaemonSet (expect 1/1 per node) ===
$ kubectl get pods -n teleport-system
NAME         READY   STATUS    RESTARTS   AGE
tbot-jnldt   1/1     Running   0          6m4s
tbot-kkkcl   1/1     Running   0          6m4s
tbot-q2ccm   1/1     Running   0          6m4s


=== Existing AuthorizationPolicies in sock-shop ===
$ kubectl get authorizationpolicies -n sock-shop
NAME                           ACTION   AGE
carts-allow-frontend           ALLOW  

## Step 1 — Create VM Workload Identity Resources in Teleport

Three Teleport resources are needed before the VM container can obtain a SPIFFE SVID:

| Resource | Purpose |
|----------|--------|
| `WorkloadIdentity/demo-vm-service` | Defines the SPIFFE ID template: `spiffe://DOMAIN/demo/vm-service` |
| `Role/demo-vm-workload-identity` | Grants the bot permission to request that specific identity |
| Bot `demo-vm-bot` + join token | The VM's Teleport identity — joins with a token (not Kubernetes) |

### 1a. WorkloadIdentity Resource

[tbot/vm-workload-identity.yaml](tbot/vm-workload-identity.yaml) tells Teleport what SPIFFE ID to issue when this
identity is requested:

```yaml
spec:
  spiffe:
    id: /demo/vm-service
    # Issues: spiffe://<trust-domain>/demo/vm-service
```

This is the SPIFFE ID that will appear in the VM's certificate, and the value we'll reference in the Istio
`AuthorizationPolicy` principal.

In [47]:
run("tctl rm workload_identity/demo-vm-service", capture=True, check=False)
run("tctl create -f tbot/vm-workload-identity.yaml", capture=True)
run("tctl get workload_identity/demo-vm-service", capture=True)

$ tctl rm workload_identity/demo-vm-service
ERROR: workload_identity "demo-vm-service" doesn't exist


$ tctl create -f tbot/vm-workload-identity.yaml
Workload identity "demo-vm-service" has been created

$ tctl get workload_identity/demo-vm-service
kind: workload_identity
metadata:
  labels:
    purpose: vm-demo
  name: demo-vm-service
  revision: b59f2667-6295-46b0-b73a-e7d8287719ed
spec:
  spiffe:
    id: /demo/vm-service
version: v1



### 1b. Bot Role

[tbot/vm-bot-role.yaml](tbot/vm-bot-role.yaml) grants the bot least-privilege access — it can only list/read
workload identities tagged `purpose: vm-demo`, which is exactly the label on `demo-vm-service`.

In [48]:
run("tctl rm role/demo-vm-workload-identity", capture=True, check=False)
run("tctl create -f tbot/vm-bot-role.yaml", capture=True)
run("tctl get role/demo-vm-workload-identity", capture=True)

$ tctl rm role/demo-vm-workload-identity
ERROR: role "demo-vm-workload-identity" is not found


$ tctl create -f tbot/vm-bot-role.yaml
role "demo-vm-workload-identity" has been created

$ tctl get role/demo-vm-workload-identity
kind: role
metadata:
  name: demo-vm-workload-identity
  revision: 07f0d3d5-0512-446c-ab85-83fae2e4fff5
spec:
  allow:
    rules:
    - resources:
      - workload_identity
      verbs:
      - list
      - read
    workload_identity_labels:
      purpose: vm-demo
  deny: {}
  options:
    cert_format: standard
    create_db_user: false
    create_desktop_user: false
    desktop_clipboard: true
    desktop_directory_sharing: true
    enhanced_recording:
    - command
    - network
    forward_agent: false
    idp:
      saml:
        enabled: true
    max_session_ttl: 30h0m0s
    pin_source_ip: false
    record_session:
      default: best_effort
      desktop: true
    ssh_file_copy: true
version: v7



### 1c. Bot and Join Token

`tctl bots add` creates the bot with the role and prints a one-time join token. tbot inside the VM container
uses this token to authenticate to Teleport on first start. After that, Teleport issues it a renewable machine
certificate — no human intervention or rotation needed.

In [49]:
run("tctl bots rm demo-vm-bot", capture=True, check=False)

result = run(
    "tctl bots add demo-vm-bot --roles=demo-vm-workload-identity --ttl=8h",
    capture=True
)

# Parse the one-time join token from the command output
match = re.search(r'(?i)(?:invite token|the bot token)[:\s]+([\w.+/=-]+)', result.stdout)
if not match:
    raise RuntimeError(
        "Could not parse bot join token from tctl output.\n"
        "Full output:\n" + result.stdout
    )

VM_BOT_TOKEN = match.group(1).strip()
print(f"\n✅ Bot token parsed (first 20 chars): {VM_BOT_TOKEN[:20]}...")
print("   (Full token stored in VM_BOT_TOKEN variable — written to .env.vm later)")

$ tctl bots rm demo-vm-bot
ERROR: user "bot-demo-vm-bot" not found


$ tctl bots add demo-vm-bot --roles=demo-vm-workload-identity --ttl=8h
The bot token: c1a14c7c55b66258e89a2b886d78c9d3
This token will expire in 479 minutes.

Optionally, if running the bot under an isolated user account, first initialize
the data directory by running the following command as root:

> tbot init \
   --destination-dir=./tbot-user \
   --bot-user=tbot \
   --reader-user=alice

... where "tbot" is the username of the bot's UNIX user, and "alice" is the
UNIX user that will be making use of the certificates.

Then, run this as the bot user to begin continuously fetching
certificates:

> tbot start \
   --destination-dir=./tbot-user \
   --token=c1a14c7c55b66258e89a2b886d78c9d3 \
   --proxy-server=ellinj.teleport.sh:443 \
   --join-method=token

Please note:

  - The ./tbot-user destination directory can be changed as desired.
  - /var/lib/teleport/bot must be accessible to the bot user, or --data-dir
    m

## Step 2 — Expose the Catalogue Service for External Access

The `catalogue` service is currently a `ClusterIP` — reachable only from within the cluster. We create a
`LoadBalancer` service (`catalogue-external`) so the Docker container on this machine can reach it.

Critically, traffic arriving via the LoadBalancer IP goes through the **pod's network namespace iptables rules**,
which Istio uses to intercept all inbound traffic and route it to the sidecar on port 15006. This means:
- The Istio sidecar **terminates** the mTLS connection
- The **`AuthorizationPolicy`** is enforced — the VM's SPIFFE ID is checked
- If allowed, the plaintext request is forwarded to the catalogue app

In [50]:
run("kubectl apply -f sockshop/catalogue-external-svc.yaml", capture=True)

print("\nWaiting for LoadBalancer IP assignment...")
CATALOGUE_HOST = None
for attempt in range(30):
    r = subprocess.run(
        "kubectl get svc catalogue-external -n sock-shop -o jsonpath='{.status.loadBalancer.ingress[0].ip}'",
        shell=True, capture_output=True, text=True, cwd=REPO_DIR
    )
    ip = r.stdout.strip().strip("'")
    if ip:
        CATALOGUE_HOST = ip
        print(f"\n✅ catalogue-external IP: {CATALOGUE_HOST}")
        break
    print(f"  waiting ({attempt+1}/30)...")
    time.sleep(5)

if not CATALOGUE_HOST:
    raise TimeoutError("catalogue-external LoadBalancer IP not assigned after 150s")

run("kubectl get svc catalogue-external -n sock-shop", capture=True)

$ kubectl apply -f sockshop/catalogue-external-svc.yaml
service/catalogue-external created


Waiting for LoadBalancer IP assignment...

✅ catalogue-external IP: 192.168.1.231
$ kubectl get svc catalogue-external -n sock-shop
NAME                 TYPE           CLUSTER-IP     EXTERNAL-IP     PORT(S)        AGE
catalogue-external   LoadBalancer   10.105.1.124   192.168.1.231   80:30428/TCP   1s



## Step 3 — Apply VM AuthorizationPolicy: Require SPIFFE ID for Catalogue Access

[sockshop/vm-catalogue-authz.yaml](sockshop/vm-catalogue-authz.yaml) adds a new `ALLOW` rule to the catalogue
service permitting the VM's SPIFFE ID:

```yaml
spec:
  selector:
    matchLabels:
      app: catalogue
  action: ALLOW
  rules:
    - from:
        - source:
            principals:
              - "TRUST_DOMAIN/demo/vm-service"   # VM's Teleport-issued SPIFFE ID
```

The rule contains no `to.operation` conditions — those would be evaluated at the HTTP layer only,
meaning the TCP-layer RBAC filter would skip the policy and fall through to `deny-all`. A
principal-only rule matches at the TCP layer where mTLS identity is available.

This is **additive** — the existing `catalogue-allow-frontend` policy still permits the front-end service.
With Istio ALLOW policies, access is granted if *any* matching ALLOW policy applies.

In [51]:
# envsubst substitutes ${TRUST_DOMAIN} in the YAML before applying
run(
    f"TRUST_DOMAIN={TELEPORT_TRUST_DOMAIN} "
    "envsubst '${TRUST_DOMAIN}' < sockshop/vm-catalogue-authz.yaml | kubectl apply -f -",
    capture=True
)

print("\n=== All catalogue AuthorizationPolicies ===")
run(
    "kubectl get authorizationpolicies -n sock-shop -o jsonpath='{range .items[?(@.spec.selector.matchLabels.app==\"catalogue\")]}{.metadata.name}\\n{end}'",
    capture=True, check=False
)
run("kubectl get authorizationpolicies -n sock-shop", capture=True)

print(f"\nVM SPIFFE ID that will be allowed: spiffe://{TELEPORT_TRUST_DOMAIN}/demo/vm-service")

$ TRUST_DOMAIN=ellinj.teleport.sh envsubst '${TRUST_DOMAIN}' < sockshop/vm-catalogue-authz.yaml | kubectl apply -f -
authorizationpolicy.security.istio.io/catalogue-allow-vm created


=== All catalogue AuthorizationPolicies ===
$ kubectl get authorizationpolicies -n sock-shop -o jsonpath='{range .items[?(@.spec.selector.matchLabels.app=="catalogue")]}{.metadata.name}\n{end}'
catalogue-allow-frontend\ncatalogue-allow-vm\n
$ kubectl get authorizationpolicies -n sock-shop
NAME                           ACTION   AGE
carts-allow-frontend           ALLOW    5m37s
carts-db-allow-carts           ALLOW    5m37s
catalogue-allow-frontend       ALLOW    5m37s
catalogue-allow-vm             ALLOW    1s
catalogue-db-allow-catalogue   ALLOW    5m37s
deny-all                                5m50s
frontend-allow-external        ALLOW    5m37s
orders-allow-frontend          ALLOW    5m37s


VM SPIFFE ID that will be allowed: spiffe://ellinj.teleport.sh/demo/vm-service


## Step 4 — Build and Start the VM Container

### What's in the container

| Component | Role |
|-----------|------|
| `tbot` | Joins Teleport with the one-time token; serves the SPIFFE Workload API on `/run/teleport/workload.sock` |
| `envoy` | Forward proxy on `:8080`; fetches SVID from the socket via SDS; initiates mTLS outbound |
| Admin UI | Envoy admin on `:9901`; shows the SVID held in memory (key demo talking point) |

The container image is built from [vm/Dockerfile](vm/Dockerfile). No secrets are baked into the image — they
arrive via environment variables in `.env.vm` at runtime, and `envsubst` substitutes them into the config
templates only in memory inside the running container.

### 4a. Write .env.vm

In [52]:
env_vm_path = os.path.join(REPO_DIR, ".env.vm")
env_vm_content = (
    f"TELEPORT_PROXY={TELEPORT_PROXY}\n"
    f"BOT_TOKEN={VM_BOT_TOKEN}\n"
    f"TRUST_DOMAIN={TELEPORT_TRUST_DOMAIN}\n"
    f"CATALOGUE_HOST={CATALOGUE_HOST}\n"
    "CATALOGUE_PORT=80\n"
)

with open(env_vm_path, "w") as f:
    f.write(env_vm_content)

print("✅ .env.vm written (gitignored)")
print(f"   TELEPORT_PROXY  = {TELEPORT_PROXY}")
print(f"   TRUST_DOMAIN    = {TELEPORT_TRUST_DOMAIN}")
print(f"   CATALOGUE_HOST  = {CATALOGUE_HOST}")
print(f"   BOT_TOKEN       = {VM_BOT_TOKEN[:20]}... (truncated)")

✅ .env.vm written (gitignored)
   TELEPORT_PROXY  = ellinj.teleport.sh:443
   TRUST_DOMAIN    = ellinj.teleport.sh
   CATALOGUE_HOST  = 192.168.1.231
   BOT_TOKEN       = c1a14c7c55b66258e89a... (truncated)


### 4b. Build the Docker Image

The build downloads:
1. The Envoy binary from `envoyproxy/envoy:v1.29-latest` (multi-arch: works on both Intel and Apple Silicon)
2. `tbot` from the Teleport CDN — version `18.7.6` to match the cluster

This takes 1–3 minutes on first build; subsequent builds use the Docker cache.

In [53]:
# Custom run with longer timeout for the build step
import subprocess as _sp

print("$ docker compose build demo-vm")
proc = _sp.run(
    "docker compose build demo-vm",
    shell=True, cwd=REPO_DIR, capture_output=True, text=True,
    timeout=600
)
if proc.stdout:
    print(proc.stdout[-3000:])  # last 3000 chars to avoid flooding output
if proc.stderr:
    print(proc.stderr[-2000:])
if proc.returncode != 0:
    raise RuntimeError(f"docker compose build failed (exit {proc.returncode})")
print("\n✅ Image built successfully")

$ docker compose build demo-vm
#1 [internal] load local bake definitions
#1 reading from stdin 636B done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 1.03kB done
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/envoyproxy/envoy:v1.29-latest
#3 ...

#4 [auth] library/ubuntu:pull token for registry-1.docker.io
#4 DONE 0.0s

#5 [auth] envoyproxy/envoy:pull token for registry-1.docker.io
#5 DONE 0.0s

#6 [internal] load metadata for docker.io/library/ubuntu:22.04
#6 DONE 0.4s

#3 [internal] load metadata for docker.io/envoyproxy/envoy:v1.29-latest
#3 DONE 0.4s

#7 [internal] load .dockerignore
#7 transferring context: 2B done
#7 DONE 0.0s

#8 [stage-1 1/8] FROM docker.io/library/ubuntu:22.04@sha256:962f6cadeae0ea6284001009daa4cc9a8c37e75d1f5191cf0eb83fe565b63dd7
#8 resolve docker.io/library/ubuntu:22.04@sha256:962f6cadeae0ea6284001009daa4cc9a8c37e75d1f5191cf0eb83fe565b63dd7 done
#8 DONE 0.0s

#9 [envoy-base 1/1] FROM docker.io/envoypr

### 4c. Start the Container and Wait for tbot

The entrypoint runs tbot, waits up to 30s for the socket, then starts Envoy. Allow ~15–20s for tbot to
authenticate to Teleport and open the Workload API.

In [54]:
# Stop any leftover container from a previous run
run("docker compose down", capture=True, check=False)

run("docker compose up -d demo-vm", capture=True)
print("Container started. Waiting for tbot to connect to Teleport...")

# Poll until the Workload API socket appears inside the container
socket_ready = False
for attempt in range(30):
    r = subprocess.run(
        "docker exec demo-vm test -S /run/teleport/workload.sock",
        shell=True, capture_output=True
    )
    if r.returncode == 0:
        socket_ready = True
        print(f"\n✅ Workload API socket ready (after ~{(attempt+1)*2}s)")
        break
    time.sleep(2)

if not socket_ready:
    print("\n❌ Socket not ready — check container logs:")
    run("docker logs demo-vm --tail=40", capture=True, check=False)
    raise TimeoutError("tbot socket did not appear within 60s")

# Show the last few log lines to confirm both services started
print("\n--- Container log (last 20 lines) ---")
run("docker logs demo-vm --tail=20", capture=True, check=False)

$ docker compose down
time="2026-05-13T10:07:10-04:00" level=warning msg="/Users/jeff/dev/rev-tech/use-cases/mwi/istio-spiffe/docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"

$ docker compose up -d demo-vm
time="2026-05-13T10:07:10-04:00" level=warning msg="/Users/jeff/dev/rev-tech/use-cases/mwi/istio-spiffe/docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Network istio-spiffe_default Creating 
 Network istio-spiffe_default Created 
 Container demo-vm Creating 
 Container demo-vm Created 
 Container demo-vm Starting 
 Container demo-vm Started 

Container started. Waiting for tbot to connect to Teleport...

✅ Workload API socket ready (after ~4s)

--- Container log (last 20 lines) ---
$ docker logs demo-vm --tail=20
[*] Socket ready (2s).
[*] Starting Envoy (forward proxy on :8080, admin on :9901)...

2026-05-13T14:07:11.313Z INFO [TBOT

## Step 5 — Demonstrate: No Credentials on Disk

This is one of the key talking points of the demo. A traditional VM approach writes certificate files to
disk — they can be stolen, need rotation procedures, and leave traces. Here, **no certificate file ever
exists on the VM's filesystem**. The SVID lives only in Envoy's process memory, fetched from tbot via
the Workload API socket.

In [55]:
print("=== Searching for certificate files inside the container ===")
r = subprocess.run(
    "docker exec demo-vm find / -name '*.crt' -o -name '*.pem' -o -name '*.key' 2>/dev/null "
    "| grep -v /proc | grep -v /sys | grep -v /usr/share",
    shell=True, capture_output=True, text=True
)

cert_files = r.stdout.strip()
if cert_files:
    print("Files found:")
    print(cert_files)
else:
    print("(none)")
    print("\n✅ No certificate files on disk — credentials exist only in memory")

=== Searching for certificate files inside the container ===
Files found:
/etc/ssl/certs/GTS_Root_R2.pem
/etc/ssl/certs/TUBITAK_Kamu_SM_SSL_Kok_Sertifikasi_-_Surum_1.pem
/etc/ssl/certs/DigiCert_Trusted_Root_G4.pem
/etc/ssl/certs/AC_RAIZ_FNMT-RCM.pem
/etc/ssl/certs/COMODO_ECC_Certification_Authority.pem
/etc/ssl/certs/DigiCert_TLS_ECC_P384_Root_G5.pem
/etc/ssl/certs/certSIGN_ROOT_CA.pem
/etc/ssl/certs/Security_Communication_RootCA3.pem
/etc/ssl/certs/Amazon_Root_CA_1.pem
/etc/ssl/certs/GlobalSign_ECC_Root_CA_-_R4.pem
/etc/ssl/certs/NAVER_Global_Root_Certification_Authority.pem
/etc/ssl/certs/emSign_Root_CA_-_G1.pem
/etc/ssl/certs/QuoVadis_Root_CA_1_G3.pem
/etc/ssl/certs/emSign_ECC_Root_CA_-_C3.pem
/etc/ssl/certs/COMODO_Certification_Authority.pem
/etc/ssl/certs/T-TeleSec_GlobalRoot_Class_2.pem
/etc/ssl/certs/OISTE_WISeKey_Global_Root_GC_CA.pem
/etc/ssl/certs/T-TeleSec_GlobalRoot_Class_3.pem
/etc/ssl/certs/ca-certificates.crt
/etc/ssl/certs/D-TRUST_BR_Root_CA_1_2020.pem
/etc/ssl/certs/SS

In [56]:
print("=== SPIFFE Workload API socket ===")
run("docker exec demo-vm ls -la /run/teleport/workload.sock", capture=True)
print("\n  tbot is serving the SPIFFE Workload API on this Unix socket.")
print("  Envoy connects to it via gRPC SDS to request the SVID in-memory.")
print("  In production, socket permissions (UID/GID) restrict which processes can request SVIDs.")

=== SPIFFE Workload API socket ===
$ docker exec demo-vm ls -la /run/teleport/workload.sock
srwxrwxrwx 1 root root 0 May 13 14:07 /run/teleport/workload.sock


  tbot is serving the SPIFFE Workload API on this Unix socket.
  Envoy connects to it via gRPC SDS to request the SVID in-memory.
  In production, socket permissions (UID/GID) restrict which processes can request SVIDs.


In [57]:
print("=== SVID held in Envoy memory (Envoy admin /certs endpoint) ===")
print("  Envoy fetched this SVID from tbot via SDS — it was never written to the filesystem.\n")

r = subprocess.run(
    "docker exec demo-vm curl -s http://localhost:9901/certs",
    shell=True, capture_output=True, text=True
)

if r.returncode != 0 or not r.stdout.strip():
    print("Envoy admin not yet responding — waiting 5s and retrying...")
    time.sleep(5)
    r = subprocess.run(
        "docker exec demo-vm curl -s http://localhost:9901/certs",
        shell=True, capture_output=True, text=True
    )

import json
try:
    certs = json.loads(r.stdout)
    # Extract SPIFFE ID lines for a clean display
    cert_text = json.dumps(certs, indent=2)
    lines = cert_text.splitlines()
    spiffe_lines = [l for l in lines if 'spiffe://' in l or 'serial' in l.lower() or 'valid' in l.lower() or 'ROOTCA' in l]
    print("Key fields from /certs:")
    for l in spiffe_lines[:20]:
        print(l)
    if not spiffe_lines:
        print(cert_text[:2000])
except Exception:
    print(r.stdout[:2000])

if f"/demo/vm-service" in r.stdout:
    print(f"\n✅ SPIFFE ID confirmed in Envoy memory: spiffe://{TELEPORT_TRUST_DOMAIN}/demo/vm-service")
else:
    print("\n⚠️  SPIFFE ID not yet visible — SDS may still be initialising. Wait 5s and re-run.")

=== SVID held in Envoy memory (Envoy admin /certs endpoint) ===
  Envoy fetched this SVID from tbot via SDS — it was never written to the filesystem.

Key fields from /certs:
          "serial_number": "f482fca07c7ae984aea5648ec07e0a5f",
          "valid_from": "2025-11-17T17:51:19Z",
          "serial_number": "e74b66ee25844ba4c3072bed563eb1e6",
              "uri": "spiffe://ellinj.teleport.sh/demo/vm-service"
          "valid_from": "2026-05-13T14:06:13Z",

✅ SPIFFE ID confirmed in Envoy memory: spiffe://ellinj.teleport.sh/demo/vm-service


## Step 6 — Test: Successful mTLS Request from VM to Catalogue

We `curl` from inside the container to Envoy's forward proxy on `localhost:8080`. Envoy:
1. Receives the plaintext request
2. Fetches the SVID from tbot via the socket (already cached in memory)
3. Opens an mTLS connection to `catalogue-external`'s LoadBalancer IP
4. The Istio sidecar validates the cert (Teleport CA, correct trust domain) and checks the `AuthorizationPolicy`
5. The policy `catalogue-allow-vm` matches → access granted → catalogue returns the product list

In [58]:
print("=== Request: VM → localhost:8080 (Envoy forward proxy) → catalogue (mTLS) ===")
print(f"   Envoy will present SPIFFE ID: spiffe://{TELEPORT_TRUST_DOMAIN}/demo/vm-service\n")

result = subprocess.run(
    "docker exec demo-vm curl -s -o /dev/null -w 'HTTP %{http_code}' http://localhost:8080/catalogue",
    shell=True, capture_output=True, text=True
)
http_code = result.stdout.strip()
print(f"Response: {http_code}")

if "200" in http_code:
    print("\n✅ Request succeeded — Istio validated the Teleport-issued SVID and AuthorizationPolicy allowed it")
    print("   Fetch the first product to confirm content:")
    run(
        "docker exec demo-vm curl -s http://localhost:8080/catalogue | python3 -c "
        "'import json,sys; items=json.load(sys.stdin); print(json.dumps(items[0], indent=2)) if items else print(\"(empty response)\")'  ",
        capture=True, check=False
    )
else:
    print(f"\n❌ Unexpected response: {http_code}")
    print("   Verbose output:")
    run("docker exec demo-vm curl -sv http://localhost:8080/catalogue 2>&1 | tail -20", capture=True, check=False)

=== Request: VM → localhost:8080 (Envoy forward proxy) → catalogue (mTLS) ===
   Envoy will present SPIFFE ID: spiffe://ellinj.teleport.sh/demo/vm-service

Response: HTTP 200

✅ Request succeeded — Istio validated the Teleport-issued SVID and AuthorizationPolicy allowed it
   Fetch the first product to confirm content:
$ docker exec demo-vm curl -s http://localhost:8080/catalogue | python3 -c 'import json,sys; items=json.load(sys.stdin); print(json.dumps(items[0], indent=2)) if items else print("(empty response)")'  
{
  "id": "03fef6ac-1896-4ce8-bd69-b798f85c6e0b",
  "name": "Holy",
  "description": "Socks fit for a Messiah. You too can experience walking in water with these special edition beauties. Each hole is lovingly proggled to leave smooth edges. The only sock approved by a higher power.",
  "imageUrl": [
    "/catalogue/images/holy_1.jpeg",
    "/catalogue/images/holy_2.jpeg"
  ],
  "price": 99.99,
  "count": 1,
  "tag": [
    "action",
    "magic"
  ]
}



## Step 7 — Live Policy Demo: DENY

We delete `catalogue-allow-vm` — the AuthorizationPolicy that permitted the VM's SPIFFE ID. The existing
`deny-all` policy immediately takes effect for the VM (the front-end is still allowed by its own policy).

**No restarts. No cert changes. No changes to the VM container at all.** One `kubectl delete` and the VM is
locked out in seconds.

The denial manifests as HTTP 503 (Envoy receives a TCP connection reset from the Istio sidecar) rather
than 403, because `deny-all` is enforced at the TCP layer before any HTTP decoding.

In [59]:
print("Deleting catalogue-allow-vm AuthorizationPolicy...")
run("kubectl delete authorizationpolicy catalogue-allow-vm -n sock-shop", capture=True)

print("\nWaiting 5s for policy to propagate to sidecars...")
time.sleep(5)

# TCP-layer RBAC denial (deny-all is evaluated before HTTP decoding) surfaces as 503
# (connection reset by Istio sidecar). HTTP 403 would only appear if the policy matched
# at the HTTP layer. Both mean "denied".
print("\n=== Request: expect 503 RBAC denied (TCP-layer connection reset) ===")
result = subprocess.run(
    "docker exec demo-vm curl -s -o /dev/null -w 'HTTP %{http_code}' http://localhost:8080/catalogue",
    shell=True, capture_output=True, text=True
)
http_code = result.stdout.strip()
print(f"Response: {http_code}")

if "503" in http_code or "403" in http_code:
    print("\n✅ VM is locked out — AuthorizationPolicy denied. No cert changes, no restarts.")
else:
    print(f"\n⚠️  Got {http_code} — policy may not have propagated yet. Wait 5s and re-run.")

Deleting catalogue-allow-vm AuthorizationPolicy...
$ kubectl delete authorizationpolicy catalogue-allow-vm -n sock-shop
authorizationpolicy.security.istio.io "catalogue-allow-vm" deleted from sock-shop namespace


Waiting 5s for policy to propagate to sidecars...

=== Request: expect 503 RBAC denied (TCP-layer connection reset) ===
Response: HTTP 503

✅ VM is locked out — AuthorizationPolicy denied. No cert changes, no restarts.


## Step 8 — Live Policy Demo: ALLOW (Restore)

Re-apply the policy to restore access. Envoy doesn't need to restart, no certs are renewed — the SVID in
memory is unchanged and will be accepted again as soon as the policy propagates.

In [60]:
print("Re-applying catalogue-allow-vm AuthorizationPolicy...")
run(
    f"TRUST_DOMAIN={TELEPORT_TRUST_DOMAIN} "
    "envsubst '${TRUST_DOMAIN}' < sockshop/vm-catalogue-authz.yaml | kubectl apply -f -",
    capture=True
)

print("\nWaiting 5s for policy to propagate...")
time.sleep(5)

print("\n=== Request: expect 200 again ===")
result = subprocess.run(
    "docker exec demo-vm curl -s -o /dev/null -w 'HTTP %{http_code}' http://localhost:8080/catalogue",
    shell=True, capture_output=True, text=True
)
http_code = result.stdout.strip()
print(f"Response: {http_code}")

if "200" in http_code:
    print("\n✅ Access restored — no restarts, no cert rotation, no VM changes.")
    print("   This is zero-trust policy enforcement in practice.")
else:
    print(f"\n⚠️  Got {http_code} — policy may still be propagating. Wait 5s and re-run.")

Re-applying catalogue-allow-vm AuthorizationPolicy...
$ TRUST_DOMAIN=ellinj.teleport.sh envsubst '${TRUST_DOMAIN}' < sockshop/vm-catalogue-authz.yaml | kubectl apply -f -
authorizationpolicy.security.istio.io/catalogue-allow-vm created


Waiting 5s for policy to propagate...

=== Request: expect 200 again ===
Response: HTTP 200

✅ Access restored — no restarts, no cert rotation, no VM changes.
   This is zero-trust policy enforcement in practice.


## Step 9 — Architecture Summary & Key Talking Points

### What Just Happened End-to-End

```
curl (in container)            Envoy (container)              catalogue pod
──────────────────             ─────────────────              ─────────────
  GET /catalogue  ──plain──►  fetch SVID from socket          
  http://localhost:8080        via SDS (in-memory)            
                               open mTLS connection  ──TLS──► port 15006
                               present SPIFFE cert            (Istio sidecar)
                                                              validate cert
                                                              check AuthzPolicy
                                                              forward to :80
```

### Key Talking Points

| Demo moment | What to show | What to say |
|-------------|-------------|-------------|
| No cert files | `find` returns nothing | "No credentials on disk — the VM has no credential files anywhere." |
| Socket | `ls -la /run/teleport/workload.sock` | "This socket is the SPIFFE Workload API. tbot serves SVIDs here. Only processes with access to this socket can request the identity." |
| Envoy admin `/certs` | SPIFFE ID in memory | "Envoy fetched this in-memory via SDS. It never touched the filesystem." |
| 200 response | Catalogue data | "Teleport issued the SVID, Istio's sidecar validated it — same CA — and the AuthorizationPolicy let it through." |
| 403 after delete | Single kubectl delete | "One command, no restarts. The VM is instantly locked out." |
| 200 after restore | Single kubectl apply | "And one command to restore. This is zero-trust policy enforcement." |
| Both K8s and VM | Same Teleport CA | "Teleport is the single identity authority for both the mesh and off-cluster workloads. One audit trail, one CA, one policy plane." |

## Step 10 — Cleanup

Removes all resources created by **this notebook only**. Resources from Part 1 (Istio, tbot DaemonSet,
Sock Shop) are left intact.

Removes:
- Docker container (`demo-vm`)
- `catalogue-external` LoadBalancer service
- `catalogue-allow-vm` AuthorizationPolicy
- Teleport: bot `demo-vm-bot`, role `demo-vm-workload-identity`, workload identity `demo-vm-service`
- Local `.env.vm` file

In [ ]:
print("=== Stopping VM container ===")
run("docker compose down", capture=True, check=False)

print("\n=== Removing Kubernetes resources ===")
run("kubectl delete svc catalogue-external -n sock-shop --ignore-not-found", capture=True)
run("kubectl delete authorizationpolicy catalogue-allow-vm -n sock-shop --ignore-not-found", capture=True)

print("\n=== Removing Teleport resources ===")
run("tctl bots rm demo-vm-bot", capture=True, check=False)
run("tctl rm workload_identity/demo-vm-service", capture=True, check=False)
run("tctl rm role/demo-vm-workload-identity", capture=True, check=False)

print("\n=== Removing local .env.vm ===")
env_vm_path = os.path.join(REPO_DIR, ".env.vm")
if os.path.exists(env_vm_path):
    os.unlink(env_vm_path)
    print("✅ .env.vm deleted")
else:
    print("   .env.vm not found (already deleted)")

print("\n=== Cleanup complete ===")
print("Part 1 resources (Istio, tbot DaemonSet, Sock Shop) are unchanged.")

## Appendix: Export Executed Notebook as HTML

Run this cell after a successful demo to save a static HTML snapshot.

In [ ]:
import sys
import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
output_file = f"demo-walkthrough-part2-executed-{timestamp}.html"

proc = _sp.run(
    f"{sys.executable} -m jupyter nbconvert --to html demo-walkthrough-part2.ipynb --output {output_file}",
    shell=True, cwd=REPO_DIR, capture_output=True, text=True, timeout=120
)
if proc.returncode == 0:
    print(f"✅ Saved: {os.path.join(REPO_DIR, output_file)}")
else:
    print("❌ Export failed:")
    print(proc.stderr)